In [1]:
import keras
from keras.applications.resnet50 import ResNet50
from keras.applications.resnet50 import preprocess_input, decode_predictions
import numpy as np

model = ResNet50(weights='imagenet')

img_path = r"/mnt/d/DL-Algorithm/CNN/image copy 2.png"
img = keras.utils.load_img(img_path, target_size=(224, 224))
x = keras.utils.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
# decode the results into a list of tuples (class, description, probability)
# (one such list for each sample in the batch)
print('Predicted:', decode_predictions(preds, top=3)[0])
# Predicted: [(u'n02504013', u'Indian_elephant', 0.82658225), (u'n01871265', u'tusker', 0.1122357), (u'n02504458', u'African_elephant', 0.061040461)]


I0000 00:00:1787933989.642008   19157 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787933991.610419   19157 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787934035.243655   19157 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1787934128.782704   19157 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted: [('n02123159', 'tiger_cat', np.float32(0.61677665)), ('n02123045', 'tabby', np.float32(0.3383027)), ('n02124075', 'Egyptian_cat', np.float32(0.017735532))]


In [2]:
import cv2 as cv
import numpy as np

print("OpenCV:", cv.__version__)
img = np.zeros((120, 400, 3), dtype=np.uint8)
cv.putText(img, "OpenCV OK", (10, 80), cv.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 3)
# If you installed a non-headless build, you can display a window:
# cv.imshow("hello", img); cv.waitKey(0)
# Always safe (headless or not): save to file
cv.imwrite("hello.png", img)

OpenCV: 5.0.0


True

In [ ]:
import cv2 as cv
import numpy as np

camera = cv.VideoCapture(0)
if not camera.isOpened():
    raise RuntimeError("Could not open the camera. Check camera permissions and try camera index 1.")

print("Camera ready. Press SPACE to capture an image or Q to cancel.")
captured_frame = None

try:
    while True:
        success, frame = camera.read()
        if not success:
            raise RuntimeError("Could not read a frame from the camera.")

        cv.imshow("Camera - SPACE to capture, Q to cancel", frame)
        key = cv.waitKey(1) & 0xFF

        if key == ord("q"):
            break
        if key == 32:
            captured_frame = frame.copy()
            break
finally:
    camera.release()
    cv.destroyAllWindows()

if captured_frame is not None:
    cv.imwrite("camera_capture.jpg", captured_frame)

    # OpenCV captures BGR images; ResNet50 expects RGB images.
    rgb_frame = cv.cvtColor(captured_frame, cv.COLOR_BGR2RGB)
    rgb_frame = cv.resize(rgb_frame, (224, 224))
    x = np.expand_dims(rgb_frame.astype("float32"), axis=0)
    x = preprocess_input(x)

    predictions = model.predict(x, verbose=0)
    print("Predicted:")
    for _, label, probability in decode_predictions(predictions, top=3)[0]:
        print(f"{label}: {probability:.2%}")
else:
    print("No image was captured.")